# Why does shape regularity matter?

The following local interpolation estimates have different geometric assumptions

$$
\begin{align*}
\|u-I_Tu\|_{L^2(T)}\le C h_T^2|u|_{H^2(T)},\qquad |u-I_Tu|_{H^1(T)}\le C_\gamma h_T|u|_{H^2(T)}.
\end{align*}
$$

The first constant is independent of the triangle's shape. The second is uniform for triangles with $\rho_T/h_T\ge\gamma>0$, where $\rho_T$ is the inradius.

We examine a smooth function on one triangle whose largest angle tends to $\pi$.

In [ ]:
from ngsolve import *
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. A flat triangle and a smooth function

Let

$$
\begin{align*}
T_\varepsilon=\operatorname{conv}\{(-1,0),(1,0),(0,\varepsilon)\}, \qquad 0<\varepsilon\le1,\qquad u(x,y)=x^2.
\end{align*}
$$

Here $h_T=2$, $|T_\varepsilon|=\varepsilon$, and

$$
\begin{align*}
\frac{\rho_T}{h_T}=\frac{\varepsilon}{2(1+\sqrt{1+\varepsilon^2})}\longrightarrow0.
\end{align*}
$$

The following helper constructs exactly one triangle.

In [ ]:
def triangle_mesh(epsilon):
    ngmesh = NetgenMesh(dim=2)
    domain = ngmesh.AddRegion("triangle", dim=2)
    vertices = [ngmesh.Add(MeshPoint(Pnt(px, py, 0)))
                for px, py in [(-1, 0), (1, 0), (0, epsilon)]]
    ngmesh.Add(Element2D(domain, vertices))
    return Mesh(ngmesh)


display_epsilons = [0.75, 0.25, 0.0625]
fig, axes = plt.subplots(1, 3, figsize=(11, 3), constrained_layout=True)
for ax, epsilon in zip(axes, display_epsilons):
    points = np.array([[-1, 0], [1, 0], [0, epsilon], [-1, 0]])
    ax.fill(points[:, 0], points[:, 1], color="C0", alpha=0.15)
    ax.plot(points[:, 0], points[:, 1], "o-", color="C0")
    ax.set(xlim=(-1.1, 1.1), ylim=(-0.1, 0.9),
           xlabel="x", ylabel="y", title=rf"$\varepsilon={epsilon:g}$")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
plt.show()

## 2. Interpolation creates a transverse slope

The three nodal values are $1$, $1$, and $0$, so

$$
\begin{align*}
I_{T_\varepsilon}u=1-\frac{y}{\varepsilon},\qquad\nabla u=(2x,0),\qquad\nabla I_{T_\varepsilon}u=(0,-1/\varepsilon).
\end{align*}
$$

The exact function is independent of $y$. Its interpolant nevertheless develops an increasingly large derivative in that direction.

For the $P_1$ space below, `Set(u_ex, dual=True)` constructs the nodal interpolant.

In [ ]:
epsilon = 0.25
mesh = triangle_mesh(epsilon)

fes = H1(mesh, order=1)
u_ex = x*x
grad_u_ex = CF((u_ex.Diff(x), u_ex.Diff(y)))
Iu = GridFunction(fes)
Iu.Set(u_ex, dual=True)

Draw(mesh, name="One triangle")
Draw(u_ex, mesh, name="Exact function x squared", deformation=True)
Draw(Iu, mesh, name="Nodal interpolant", deformation=True)

Draw(grad_u_ex[1], mesh, name="Exact y derivative", deformation=True)
Draw(grad(Iu)[1], mesh, name="Interpolant y derivative", deformation=True)
print(f"Exact y derivative: 0; interpolant y derivative: {-1/epsilon:g}")

## 3. Compare normalized errors

Since the area also shrinks, raw errors alone mix geometric scaling with the deterioration of the interpolation constant. We therefore compute

$$
\begin{align*}
C_{L^2}(\varepsilon)=\frac{\|u-I_Tu\|_{L^2(T)}}{h_T^2|u|_{H^2(T)}},\qquad C_{H^1}(\varepsilon)=\frac{|u-I_Tu|_{H^1(T)}}{h_T|u|_{H^2(T)}}.
\end{align*}
$$

These are error ratios for this particular function, hence lower bounds for any constants valid for all $u$ (They are not the optimal constants of the estimates.).

Only $\partial_{xx}u=2$ contributes to $|u|_{H^2(T)}$, so $|u|_{H^2(T)}^2=4|T|=4\varepsilon$. The integrands below are polynomials and can be integrated exactly by sufficiently accurate quadrature.

In [ ]:
epsilons = 2.0**(-np.arange(0, 9))
L2_errors, energy_errors = [], []
C_L2, C_H1, shape_ratios = [], [], []

for epsilon in epsilons:
    mesh = triangle_mesh(epsilon)
    fes = H1(mesh, order=1)
    interpolant = GridFunction(fes)
    interpolant.Set(u_ex, dual=True)
    gradient_error = grad_u_ex-grad(interpolant)
    L2_error = sqrt(Integrate((u_ex-interpolant)**2, mesh, order=4))
    energy_error = sqrt(Integrate(gradient_error**2, mesh, order=4))
    H2_seminorm = sqrt(Integrate(CF(4), mesh))
    h_T = 2

    L2_errors.append(L2_error)
    energy_errors.append(energy_error)
    C_L2.append(L2_error/(h_T**2*H2_seminorm))
    C_H1.append(energy_error/(h_T*H2_seminorm))
    shape_ratios.append(epsilon/(2*(1+np.sqrt(1+epsilon**2))))

print(" epsilon      rho / h       normalized L2     normalized H1")
for epsilon, gamma, c0, c1 in zip(epsilons, shape_ratios, C_L2, C_H1):
    print(f"{epsilon:9.5f}   {gamma:10.4e}      {c0:10.6f}        {c1:10.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].loglog(1/epsilons, C_L2, "o-", label=r"$C_{L^2}(\varepsilon)$")
axes[0].loglog(1/epsilons, C_H1, "s-", label=r"$C_{H^1}(\varepsilon)$")
axes[0].loglog(1/epsilons, 1/(4*epsilons), "k--",
               label=r"reference $1/(4\varepsilon)$")
axes[0].set(xlabel=r"$1/\varepsilon$", ylabel="normalized error",
            title="Interpolation constants")
axes[1].loglog(1/epsilons, shape_ratios, "o-", color="C2")
axes[1].set(xlabel=r"$1/\varepsilon$", ylabel=r"$\rho_T/h_T$",
            title="Loss of shape regularity")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
axes[0].legend()
plt.show()

### The mechanism in formulas

Direct integration gives

$$
\begin{align*}
\|u-I_Tu\|_{L^2(T)}^2=\frac{3\varepsilon}{10},\qquad |u-I_Tu|_{H^1(T)}^2=\frac{2\varepsilon}{3}+\frac1\varepsilon.
\end{align*}
$$

Consequently,

$$
\begin{align*}
C_{L^2}(\varepsilon)=\frac{\sqrt{3/10}}8,\qquad C_{H^1}(\varepsilon)=\frac14\sqrt{\frac23+\frac1{\varepsilon^2}}.
\end{align*}
$$

The $L^2$ ratio stays constant while the $H^1$ ratio tends to infinity. This is the effect of the inverse transformation matrix in the gradient estimate.

## Observe

- The exact function stays smooth. Which derivative of the interpolant causes the loss?
- Why would plotting only the unnormalized $L^2$ error be misleading?
- What happens to these two normalized ratios if a triangle is uniformly scaled instead of flattened?

**Takeaway.** Uniform shrinking preserves shape regularity. Flattening does not. The example demonstrates why a geometry-independent $H^1$ interpolation constant cannot hold for arbitrary triangles, however, it does not say that every thin triangle gives a large error for every function.

[← Interpolation and Galerkin](01_interpolation_and_galerkin.ipynb) · [Next: corner singularity →](03_corner_singularity.ipynb)

[Lecture notebooks](../README.md) · [Lecture notes](../../script/README.md) · [Course overview](../../index.ipynb)